# Object Detection: Bounding Box Geometry, IoU, NMS & YOLO Grid Heads

This interactive lab covers:
1. **Bounding Box Representations**: Corner $(x_1, y_1, x_2, y_2)$ vs. Center-Size $(c_x, c_y, w, h)$.
2. **Intersection over Union (IoU)**: Vectorized pairwise box overlap calculations.
3. **Non-Maximum Suppression (NMS)**: Eliminating redundant candidate boxes via greedy IoU thresholding.
4. **Single-Stage YOLO Detection Head**: Grid-based box parameterization and offset decoding.
5. **Mean Average Precision (mAP)**: Precision-Recall curves and interpolated Area-Under-Curve (AUC).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('04-computer-vision/object-detection/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import torch
from detection_engine import (
    box_cxcywh_to_xyxy,
    box_xyxy_to_cxcywh,
    compute_iou,
    non_maximum_suppression,
    MiniYOLOHead,
    compute_average_precision,
)

print('Object Detection engine loaded successfully!')

## 1. Box Geometry & Intersection-over-Union (IoU)

Converting between coordinate systems and computing overlap metrics between bounding boxes.

In [ ]:
# Ground Truth box: [10, 10, 50, 50]
gt_box = torch.tensor([[10.0, 10.0, 50.0, 50.0]])

# Two candidate predictions
pred_boxes = torch.tensor([
    [12.0, 10.0, 48.0, 52.0],  # Close match
    [30.0, 30.0, 80.0, 80.0],  # Partial overlap
    [100.0, 100.0, 150.0, 150.0] # Disjoint
])

ious = compute_iou(gt_box, pred_boxes)
print(f'IoU with Candidate 0 (Close):    {ious[0, 0].item():.4f}')
print(f'IoU with Candidate 1 (Partial):  {ious[0, 1].item():.4f}')
print(f'IoU with Candidate 2 (Disjoint): {ious[0, 2].item():.4f}')

## 2. Non-Maximum Suppression (NMS)

Filtering overlapping candidate boxes to suppress redundant detections around a single object.

In [ ]:
candidate_boxes = torch.tensor([
    [10.0, 10.0, 50.0, 50.0],  # Detection A (Score: 0.92)
    [12.0, 11.0, 49.0, 51.0],  # Detection B (Score: 0.85, Overlaps A)
    [11.0, 9.0, 52.0, 48.0],   # Detection C (Score: 0.70, Overlaps A)
    [120.0, 80.0, 160.0, 120.0], # Detection D (Score: 0.88, Other object)
])
scores = torch.tensor([0.92, 0.85, 0.70, 0.88])

kept_indices = non_maximum_suppression(candidate_boxes, scores, iou_threshold=0.5)
print(f'Indices kept after NMS: {kept_indices.tolist()}')
print(f'Suppressed redundant boxes: {[i for i in range(len(scores)) if i not in kept_indices.tolist()]}')

## 3. YOLO Single-Stage Grid Detection Head

Predicting bounding boxes, objectness, and class probabilities across an $S \times S$ spatial grid.

In [ ]:
torch.manual_seed(42)
grid_size = 4
num_classes = 3
yolo_head = MiniYOLOHead(in_channels=32, grid_size=grid_size, num_anchors=2, num_classes=num_classes)

# Feature map from backbone: (B=1, C=32, S=4, S=4)
feat = torch.randn(1, 32, grid_size, grid_size)
pred = yolo_head(feat)

print(f'Decoded Box Tensor shape:    {pred["boxes_xyxy"].shape}')
print(f'Objectness confidence shape: {pred["objectness"].shape}')
print(f'Class probabilities shape:   {pred["class_probs"].shape}')
print(f'Max objectness score in grid: {pred["objectness"].max().item():.4f}')

## 4. Evaluation: Average Precision (AP)

Computing interpolated Area Under the Precision-Recall Curve (AUC-PR).

In [ ]:
recalls = np.array([0.1, 0.25, 0.5, 0.75, 0.9, 1.0])
precisions = np.array([1.0, 0.92, 0.88, 0.81, 0.65, 0.40])

ap = compute_average_precision(precisions, recalls)
print(f'Average Precision (AP): {ap:.4f}')